# Neu — OCCOR-01 «ОЦЕНЩИК-КОРРЕЛЯЦИЯ», пилот (Colab)

1. **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить** (Runtime → Change runtime type → T4 GPU).
2. **Среда выполнения → Выполнить все** (Runtime → Run all).
3. Когда Colab попросит доступ к Google Диску — разрешить.

Идёт около 40–60 минут: по очереди скачиваются 11 моделей (Pythia, четыре чекпойнта pythia-2.8b, OPT, GPT-Neo) и отвечают на 2699 задач сложения «трёхзначное + двузначное»; затем анализ пилота (около 10 минут). Вкладку можно не трогать, но не закрывать.

Результат — папка **Neu_colab/occor_pilot** на вашем Диске (ответы пишутся туда после каждой модели). Когда в последней ячейке появится «готово», напишите Claude «готово». Личных данных ноутбук не пишет.

In [ ]:
# 1. Видеокарта и Google Диск (Colab попросит разрешение)
import os, torch
assert torch.cuda.is_available(), 'Нет видеокарты: Среда выполнения → Сменить среду выполнения → T4 GPU, затем «Выполнить все» заново'
print('видеокарта:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Neu_colab/occor_pilot'
os.makedirs(OUT, exist_ok=True)
print('папка результата:', OUT)

In [ ]:
# 2. Код опыта с GitHub — закреплённый коммит, сверка sha256
import hashlib, urllib.request
COMMIT = '2d7c7ec3c4e69ffd6dda63058b8fa14b42ea37be'
SHA = {"experiments/2026-11-14_OCCOR-01/occor_extract.py": "28f0ff5389fb5ddc734766dae28d196b6319729b9bcb175b600c61109a92f138", "experiments/2026-11-14_OCCOR-01/analyze_occor.py": "30f68c22b22d6c0d6fba42f938d418fe1be66a0ca2c6611f1b2229bb2590bb03", "experiments/2026-11-14_OCCOR-01/pilot_occor.py": "45a8484a88527162e08099ee7fb5cd6446f9455f7ebde8a5dc226f962940b27d", "experiments/2026-11-14_OCGEO-01/analyze_ocgeo.py": "d7724c385718ec5fade09cf385b69edd730230cd8423b2bc87274a2c9214ffca", "experiments/2026-11-14_OCGEO-01/analyze_ocgeo_test.py": "2aee14122df8bc287fe4a49b33dafa98c76ce3c85230ece2cbd41a03c85194ca"}
for path, want in SHA.items():
    data = urllib.request.urlopen(f'https://raw.githubusercontent.com/131ymm-commits/Neu/{COMMIT}/{path}').read()
    assert hashlib.sha256(data).hexdigest() == want, f'sha256 не совпал: {path}'
    dst = '/content/neu/' + path
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    open(dst, 'wb').write(data)
    print(path, 'sha256 совпал')

In [ ]:
# 3. Ответы голов (долго: около 40–60 минут; ответы пишутся на Диск после каждой модели)
!pip -q install accelerate
%cd {OUT}
!python /content/neu/experiments/2026-11-14_OCCOR-01/occor_extract.py

In [ ]:
# 4. Анализ пилота (около 10 минут) и запись на Диск
!python /content/neu/experiments/2026-11-14_OCCOR-01/pilot_occor.py {OUT}/pilot_out --boot 1000 --boot-dose 300
import shutil
src = '/content/neu/experiments/2026-11-14_OCCOR-01/analysis_pilot.json'
shutil.copy(src, OUT + '/analysis_pilot.json')
shutil.copy(src, OUT + '/analysis_pilot.txt')
drive.flush_and_unmount()
print('готово: записано в Neu_colab/occor_pilot — напишите Claude «готово»')